> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board and **switches the
> relay** on I2C bus 0: it clicks, and whatever is wired to its contacts switches with it.
> *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against a fake part, writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` (the first cells of
`# Hardware validation`) runs in ordinary Python (CPython) on this computer, with no board: it
writes the fake part (`qwiic_relay_fake.py`), the driver (`qwiic_relay.py`) and their tests,
and checks the driver against the fake, in real and in simulated time. After `%board connect`,
cells run in MicroPython on the board, except `%%host` cells, which run on the computer. That
section uploads the freshly written package, checks the real relay on bus 0 with a matching
test for every unit test above, then disconnects.

micropylab has `%%extract` built in, so there is no `%load_ext jupyter_lab_extractor` cell.

# Hardware and configuration
What this notebook is validated with. The same table opens `# Hardware validation`.

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0 (v1.28.0, 2026-04-06), USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(0)`, **bus 0**, the STEMMA QT connector (scl=9 sda=8, on-board pull-ups), 100 kHz (`bus_open("feathers3", 0)`) |
| **Mux** | none | none: the relay is on bus 0 directly |
| **Part** | `FakeQwiicRelay` at `0x18`: open, firmware version `0x01` | SparkFun Qwiic Single Relay at `0x18` (ADR jumper open). **Nothing may be wired to its contacts** while the hardware section runs |
| **Other parts used** | none | none. Also on bus 0: VL53L4CD at `0x29` (not used here) |
| **Datasheet** | none: the command values are from SparkFun's Qwiic_Relay library | same |
| **Result** | not run yet | not yet run on the hardware |

# TODO:
- Run the hardware section and fill in the results (this notebook has not been run on the
  hardware yet). The driver itself was used on bus 0 on 2026-11-13, before this notebook.

Decided (2026-11-16): this notebook replaces the hand-written
`src/micropython_i2c/devices/drivers/qwiic_relay.py`. The driver is ported unchanged (same API
and behaviour). Its test moved from `tests/test_drivers.py` to `tests/test_qwiic_relay.py`, one
behaviour per test. `FakeQwiicRelay` replaces `micropython_i2c.fakes.FakeRelay`, with the same
behaviour; its `closed` attribute stays public, so another fake's signal can depend on the
relay (`lambda t: 30 if relay.closed else 20`).

There is no datasheet for the relay board's firmware in `docs/datasheets/`. The command
values (`0x00` off, `0x01` on, `0x04` firmware version, `0x05` status) and the way the state is
read (write the command byte, read one byte) are from SparkFun's Qwiic_Relay library.

# Python file
location: `src/micropython_i2c/devices/drivers/qwiic_relay.py` (driver) and
`src/micropython_i2c/devices/drivers/qwiic_relay_fake.py` (fake part, host only)

tests: `tests/test_qwiic_relay.py` (both)

## header

In [ ]:
%%extract ../../src/micropython_i2c/devices/drivers/qwiic_relay.py
"""SparkFun Qwiic Single Relay (ATtiny firmware), command-per-byte protocol.

Usage on the board::

    from micropython_i2c import I2CConnection
    from micropython_i2c.devices.drivers.qwiic_relay import QwiicRelay

    relay = QwiicRelay(I2CConnection(bus, 0x18, lock=bus_lock))
    relay.on()
    print(relay.state)              # True
    relay.off()

The firmware takes one-byte commands. State and version are read the way SparkFun's
libraries do it: write the command byte, then read one byte back (an SMBus "read byte"
with the command as the register number). Command values follow SparkFun's
Qwiic_Relay library for the single-relay board.

Generated by ``device_notebooks/drivers/qwiic_relay_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None: the driver runs on MicroPython and uses only this package.

## Internal
`RegisterI2CBase` for the one-byte reads ("read byte" with the command as the register) and
the raw command writes.

In [ ]:
%%extract ../../src/micropython_i2c/devices/drivers/qwiic_relay.py -a
from micropython_i2c.node import I2CDevice
from micropython_i2c.protocol import RegisterI2CBase

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../../tests/test_qwiic_relay.py
import pytest

from micropython_i2c import I2CConnection
from micropython_i2c.devices.drivers.qwiic_relay import QwiicRelay
from micropython_i2c.devices.drivers.qwiic_relay_fake import FakeQwiicRelay
from micropython_i2c.sim.bus import FakeBus
from micropython_i2c.sim.clock import SimClock
from micropython_i2c.sim.signals import lag, value

Notebook-only: the imports the validation and test cells use. The classes under test
(`FakeQwiicRelay`, `QwiicRelay`) are the ones defined in this notebook, not the modules'.

In [ ]:
import pytest

from micropython_i2c import I2CConnection
from micropython_i2c.sim.bus import FakeBus
from micropython_i2c.sim.clock import SimClock
from micropython_i2c.sim.signals import lag, value

# Intent
**Purpose:** switch a SparkFun Qwiic Single Relay on and off over I2C, and read its state and
firmware version.

**Why it exists:** the relay board is an ATtiny with its own small command set, not a register
map: a write of one byte is a command, and a state is read by sending the command byte and
reading one byte back. The driver names the commands and always reads the state fresh from the
board, so it never reports a cached state the relay is not in.

**Used by:** the routing table and manager (`"driver": "QwiicRelay"`) and the bench (bus 0, the
STEMMA QT connector, with the VL53L4CD).

**Not its job:** the address-change command (`0x03`; the driver allows only the jumper
addresses `0x18` and `0x19`), the multi-relay boards' commands, and anything wired to the
contacts.

```python
relay = QwiicRelay(I2CConnection(bus, 0x18, lock=bus_lock))
relay.on()                 # command 0x01: close
relay.state                # command 0x05, read one byte: True
relay.toggle()             # read, flip, read back: False
relay.state = True         # same as on()
relay.firmware_version()   # command 0x04, read one byte
```

| Command | Byte | Reply |
|---|---|---|
| relay off (open) | `0x00` | none |
| relay on (closed) | `0x01` | none |
| firmware version | `0x04` | one byte |
| status | `0x05` | one byte, non-zero when closed |

## What the module defines

| Name | What it is |
|---|---|
| `QwiicRelay` | the driver: `on()`, `off()`, `toggle()`, `state` (read fresh, settable), `firmware_version()` |

The fake part, `FakeQwiicRelay`, is in its own module, `qwiic_relay_fake.py` (host only): see
`# Fake part`.

# Fake part
`FakeQwiicRelay` stands in for the real board on a `FakeBus`, so the driver can be written and
tested on the host. It is written before the driver because the driver's tests need it. It goes
to its own module next to the driver, `qwiic_relay_fake.py`, which the board never imports.

## header

In [ ]:
%%extract ../../src/micropython_i2c/devices/drivers/qwiic_relay_fake.py
"""``FakeQwiicRelay``: a SparkFun Qwiic Single Relay, for host tests.

Usage on the host::

    from micropython_i2c.devices.drivers.qwiic_relay_fake import FakeQwiicRelay
    from micropython_i2c.sim.bus import FakeBus
    from micropython_i2c.sim.signals import lag

    bus = FakeBus()
    relay = bus.attach(0x18, FakeQwiicRelay())
    heater_c = lag(lambda t: 30.0 if relay.closed else 20.0, tau_s=120)

``closed`` is public on purpose: another fake's quantity can be a signal that reads it, so a
simulated relay can switch a simulated heater.

Host only.

Generated by ``device_notebooks/drivers/qwiic_relay_devNB``: edit the notebook, not this file.
"""

## Imports
None: the fake is plain Python. It has no quantities that change with time.

## Public names

In [ ]:
%%extract ../../src/micropython_i2c/devices/drivers/qwiic_relay_fake.py -a
__all__ = ["FakeQwiicRelay"]

## `FakeQwiicRelay`
What it models, and where each behaviour comes from:

| Behaviour | Source |
|---|---|
| a one-byte write is a command: `0x01` closes the relay, `0x00` opens it, any other command leaves it as it is | SparkFun Qwiic_Relay library (single relay) |
| a "read byte" (`readfrom_mem`) with command `0x05` returns 1 when closed, 0 when open; with `0x04` the firmware version | SparkFun Qwiic_Relay library |
| the same reply after a command write and a separate one-byte read | SparkFun Qwiic_Relay library: it writes the command, ends the transfer, then reads |
| a read for any other command is refused (`OSError`) | the fake's choice: the firmware's reply is not documented |

Not modelled: the address-change command (`0x03`), the relay's switching time and contact
bounce, and the multi-relay boards' commands.

Same behaviour as the old `micropython_i2c.fakes.FakeRelay`, plus the separate read and a
`switches` count. The old fake raised `KeyError` for an unknown read; this one raises
`OSError`, as a NACK would.

In [ ]:
%%extract ../../src/micropython_i2c/devices/drivers/qwiic_relay_fake.py -a
class FakeQwiicRelay:
    """SparkFun Qwiic Single Relay: one-byte commands, status read as a command byte.

    Args:
        version: The firmware version byte (command ``0x04``).

    Attributes:
        closed: True while the relay is closed (on). Public: signals may read it.
        version: The firmware version byte.
        switches: How many times the relay has changed state.
        command: The last command byte written (None before the first).
    """

    CMD_OFF = 0x00
    CMD_ON = 0x01
    CMD_FIRMWARE_VERSION = 0x04
    CMD_STATUS = 0x05

    def __init__(self, version=0x01):
        self.closed = False
        self.version = version
        self.switches = 0
        self.command = None

    def _reply(self, cmd):
        if cmd == self.CMD_STATUS:
            return int(self.closed)
        if cmd == self.CMD_FIRMWARE_VERSION:
            return self.version
        raise OSError(5, "EIO: no reply for command {}".format(cmd))

    def raw_write(self, data):
        self.command = data[0]
        if data[0] in (self.CMD_OFF, self.CMD_ON):
            closed = data[0] == self.CMD_ON
            if closed != self.closed:
                self.switches += 1
            self.closed = closed

    def raw_read(self, nbytes):
        return bytes([self._reply(self.command)]) * nbytes

    def mem_read(self, reg, nbytes):
        self.command = reg
        return bytes([self._reply(reg)]) * nbytes

### validation/test

#### validation: nominal usage
Commands straight on the bus: close, read the status both ways, open, read the version.

In [ ]:
bus = FakeBus()
part = bus.attach(0x18, FakeQwiicRelay(version=0x02))
bus.writeto(0x18, b"\x01")
print("after 0x01: closed {}, status byte {}".format(part.closed, bus.readfrom_mem(0x18, 0x05, 1)))
bus.writeto(0x18, b"\x05")
print("status by write-then-read:", bus.readfrom(0x18, 1))
bus.writeto(0x18, b"\x00")
print("after 0x00: closed {}, version byte {} | switches {}".format(
    part.closed, bus.readfrom_mem(0x18, 0x04, 1), part.switches))  # fmt: skip

#### validation: anti-nominal usage
A read for a command with no reply is refused, as a NACK.

In [ ]:
try:
    bus.readfrom_mem(0x18, 0x07, 1)  # 0x07: no such command
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a read for command 0x07 should have raised OSError")
finally:
    print("relay still open:", not part.closed)

#### unit

`0x01` closes and `0x00` opens; other commands leave the relay alone; `switches` counts only
real changes.

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_fake_switches_on_commands():
    bus = FakeBus()
    part = bus.attach(0x18, FakeQwiicRelay())
    for cmd, closed in ((0x01, True), (0x01, True), (0x04, True), (0x00, False), (0x05, False)):
        bus.writeto(0x18, bytes([cmd]))
        assert part.closed is closed
    assert part.switches == 2

The status and the version read the same as a "read byte" and as a command write followed by a
separate read.

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_fake_reads_status_and_version_both_ways():
    bus = FakeBus()
    bus.attach(0x18, FakeQwiicRelay(version=0x07))
    bus.writeto(0x18, b"\x01")
    assert bus.readfrom_mem(0x18, 0x05, 1) == b"\x01"
    assert bus.readfrom_mem(0x18, 0x04, 1) == b"\x07"
    bus.writeto(0x18, b"\x05")
    assert bus.readfrom(0x18, 1) == b"\x01"

A read for a command with no reply is refused with `OSError`.

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_fake_refuses_a_read_without_a_reply():
    bus = FakeBus()
    bus.attach(0x18, FakeQwiicRelay())
    with pytest.raises(OSError):
        bus.readfrom_mem(0x18, 0x07, 1)
    with pytest.raises(OSError):
        bus.readfrom(0x18, 1)  # no command written yet

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../../src/micropython_i2c/devices/drivers/qwiic_relay.py -a
__all__ = ["QwiicRelay"]

## `QwiicRelay`
The driver, ported unchanged. `on()` and `off()` write one command byte (`write_raw`); `state`
and `firmware_version()` are one-byte "read byte" transfers with the command as the register
(`reg_read`, 8-bit register, one byte). `state` is never cached: every read goes to the board.
`toggle()` reads the state, writes the opposite, and returns the state read back.

The address is `0x18`, or `0x19` with the ADR jumper closed. The firmware can be given another
address by command; `ALLOWED_ADDRS` would then need widening. 100 kHz nominal, 400 kHz at most.

No method raises on its own: a missing or silent board shows as the bus's `OSError`.

In [ ]:
%%extract ../../src/micropython_i2c/devices/drivers/qwiic_relay.py -a
class QwiicRelay(RegisterI2CBase, I2CDevice):
    """SparkFun Qwiic Single Relay: ``on()``, ``off()``, ``toggle()``, ``state``."""

    # 0x18 by default, 0x19 with the ADR jumper closed. The firmware can also be given
    # another address by command; widen this tuple if you do that.
    ALLOWED_ADDRS = (0x18, 0x19)
    BAUD_NOMINAL = 100_000
    BAUD_MAX = 400_000
    BAUD_MIN = 10_000

    REG_BYTES = 1

    CMD_OFF = 0x00
    CMD_ON = 0x01
    CMD_FIRMWARE_VERSION = 0x04
    CMD_STATUS = 0x05

    def on(self):
        """Close the relay."""
        self.write_raw(bytes([self.CMD_ON]))

    def off(self):
        """Open the relay."""
        self.write_raw(bytes([self.CMD_OFF]))

    @property
    def state(self):
        """True if the relay is closed. Always read fresh from the board."""
        return bool(self.reg_read(self.CMD_STATUS))

    @state.setter
    def state(self, closed):
        if closed:
            self.on()
        else:
            self.off()

    def toggle(self):
        """Flip the relay. Returns the new state as read back."""
        self.state = not self.state
        return self.state

    def firmware_version(self):
        """The firmware version byte."""
        return self.reg_read(self.CMD_FIRMWARE_VERSION)

### validation/test: the driver on a fake part
A helper that puts a `FakeQwiicRelay` at `0x18` on a fake bus and binds the driver to it
(keywords go to the fake). It returns the driver and the fake part, and is shared by the
validation cells and the test file.

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a
def on_fake(addr=0x18, **part_kw):
    """``(relay, part)``: a ``QwiicRelay`` on a fresh ``FakeQwiicRelay(**part_kw)`` at ``addr``."""
    bus = FakeBus()
    part = bus.attach(addr, FakeQwiicRelay(**part_kw))
    return QwiicRelay(I2CConnection(bus, addr)), part

#### validation: nominal usage

In [ ]:
relay, part = on_fake(version=0x02)
print("at start: state {}, version {:#04x}".format(relay.state, relay.firmware_version()))
relay.on()
print("after on(): state {} | fake closed {}".format(relay.state, part.closed))
print("toggle() ->", relay.toggle())
relay.state = True
print("after state = True: fake closed {}, switches {}".format(part.closed, part.switches))
relay.off()

#### validation: anti-nominal usage
The driver has no failure path of its own: a board that does not answer shows as the bus's
`OSError`. Here the board has its ADR jumper closed (`0x19`) and the driver talks to `0x18`.

In [ ]:
bus = FakeBus()
part = bus.attach(0x19, FakeQwiicRelay())
relay = QwiicRelay(I2CConnection(bus, 0x18))  # the board is at 0x19
try:
    relay.on()
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a relay at the wrong address should have raised OSError")
finally:
    print("the board at 0x19 is still open:", not part.closed)

#### unit

`on()` closes and `off()` opens, and `state` reads it (ported from `tests/test_drivers.py`).

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_on_off_and_state():
    relay, part = on_fake()
    assert relay.state is False
    relay.on()
    assert part.closed and relay.state is True
    relay.off()
    assert not part.closed and relay.state is False

`toggle()` flips the relay and returns the new state (ported).

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_toggle_returns_the_new_state():
    relay, part = on_fake()
    relay.on()
    assert relay.toggle() is False and not part.closed
    assert relay.toggle() is True and part.closed

Setting `state` switches the relay (ported).

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_state_setter():
    relay, part = on_fake()
    relay.state = True
    assert part.closed
    relay.state = False
    assert not part.closed

`state` is read fresh from the board: a change made behind the driver's back shows at once.

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_state_is_read_fresh():
    relay, part = on_fake()
    part.closed = True  # someone else switched it
    assert relay.state is True

`firmware_version()` returns the version byte (ported).

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_firmware_version():
    relay, _ = on_fake(version=0x02)
    assert relay.firmware_version() == 0x02

Only the two jumper addresses are allowed, and the clock limit is 400 kHz.

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_address_and_clock_limits():
    assert QwiicRelay.addr_ok(0x18) and QwiicRelay.addr_ok(0x19)
    assert not QwiicRelay.addr_ok(0x20)
    assert QwiicRelay.baud_ok(400_000) and not QwiicRelay.baud_ok(1_000_000)

### validation/test: in simulated time
A thermostat: the relay switches a heater, and the room temperature is a `lag` signal (time
constant 120 s) towards 30 C while the relay is closed and 18 C while it is open. The signal
reads the fake's public `closed`. Inside `with SimClock() as clock:` a control loop reads the
temperature every 5 simulated seconds and switches with the driver: on below 21.5 C, off above
22.5 C. An hour runs in a fraction of a second.

This block has no matching hardware test: there is no heater on the bench.

Notebook-only: the thermostat loop, shared by the validation cell below. The unit test runs
the same loop inline, so the test file needs nothing from here.

In [ ]:
def thermostat(relay, temp, clock, minutes, on_below=21.5, off_above=22.5, every_s=5):
    """Run the control loop; returns ``[(t_s, temp_c, closed), ...]``, one per step."""
    log = []
    for _ in range(int(minutes * 60 / every_s)):
        t_c = value(temp)
        if t_c < on_below:
            relay.on()
        elif t_c > off_above:
            relay.off()
        log.append((clock.now_s(), t_c, relay.state))
        clock.advance(every_s)
    return log

#### validation: nominal usage
One hour. The first 4 minutes are printed every 20 s (the warm-up and the first switching),
then the range the temperature kept to after 10 minutes.

In [ ]:
with SimClock() as clock:
    relay, part = on_fake()
    temp = lag(lambda t: 30.0 if part.closed else 18.0, tau_s=120, initial=18.0)
    log = thermostat(relay, temp, clock, minutes=60)
for t_s, t_c, closed in log[:48:4]:
    print("t = {:3.0f} s: {:5.2f} C, relay {}".format(t_s, t_c, "on" if closed else "off"))
late = [t_c for t_s, t_c, _ in log if t_s >= 600]
print("after 10 min: {:.2f} .. {:.2f} C | relay switched {} times".format(
    min(late), max(late), part.switches))  # fmt: skip

#### validation: anti-nominal usage
With the heater too weak (its target 21 C, below the switch-off point), the relay closes once
and never opens again: the loop cannot reach 22.5 C.

In [ ]:
with SimClock() as clock:
    relay, part = on_fake()
    temp = lag(lambda t: 21.0 if part.closed else 18.0, tau_s=120, initial=18.0)
    log = thermostat(relay, temp, clock, minutes=60)
print("final {:.2f} C, relay {}, switched {} time(s)".format(
    log[-1][1], "on" if log[-1][2] else "off", part.switches))  # fmt: skip
if part.switches != 1 or not part.closed:
    raise AssertionError("a heater that cannot reach the set point should stay on")

#### unit

The relay holds the room between about 21 and 23 C after the first 10 minutes, switching both
ways many times.

In [ ]:
%%extract ../../tests/test_qwiic_relay.py -a --strip-ipytest
%%ipytest


def test_thermostat_holds_the_band_in_simulated_time():
    with SimClock() as clock:
        relay, part = on_fake()
        temp = lag(lambda t: 30.0 if part.closed else 18.0, tau_s=120, initial=18.0)
        late = []
        for _ in range(720):  # one hour, every 5 s
            t_c = value(temp)
            if t_c < 21.5:
                relay.on()
            elif t_c > 22.5:
                relay.off()
            if clock.now_s() >= 600:
                late.append(t_c)
            clock.advance(5)
    assert 21.0 <= min(late) and max(late) <= 23.0
    assert part.switches >= 10 and part.closed == relay.state

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, and **switches the Qwiic Single Relay on and off** on I2C bus 0. The relay clicks,
> and anything wired to its contacts (COM, NO, NC) switches with it.
>
> - Before you start: **disconnect everything from the relay's screw terminals**, or make sure
>   that switching the load is harmless. Never run this with mains on the contacts.
> - Do **not** use *Run All*, *Run All Below* or `jlx run` on this section and walk away.
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - Each cell leaves the relay **open (off)** when it finishes, even if it fails.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

**Hardware and configuration** (same as the table at the top):

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0 (v1.28.0, 2026-04-06), USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(0)`, **bus 0**, the STEMMA QT connector (scl=9 sda=8, on-board pull-ups), 100 kHz (`bus_open("feathers3", 0)`) |
| **Mux** | none | none: the relay is on bus 0 directly |
| **Part** | `FakeQwiicRelay` at `0x18`: open, firmware version `0x01` | SparkFun Qwiic Single Relay at `0x18` (ADR jumper open). **Nothing may be wired to its contacts** while the hardware section runs |
| **Other parts used** | none | none. Also on bus 0: VL53L4CD at `0x29` (not used here) |
| **Datasheet** | none: the command values are from SparkFun's Qwiic_Relay library | same |
| **Result** | not run yet | not yet run on the hardware |

Layout: connect and set up, find the part, then **matching tests** (one per unit test above,
same name, run on the real board and on MicroPython; the fake's own tests are matched where
the real board can show the same thing), a hardware-only check, then the tally and disconnect.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it, and that
nothing is wired to the relay's contacts.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c import I2CConnection, default_lock
from micropython_i2c.boards import bus_open
from micropython_i2c.compat import sleep_ms
from micropython_i2c.debug import check, check_reset, check_summary, info
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.devices.drivers.qwiic_relay import QwiicRelay

check_reset()

Notebook-only: the bus and helpers. `relay_conn()` is a connection to the relay; `rest()` opens
the relay with a raw command (whatever the driver does), and every cell that switches it calls
`rest()` in its `finally`. `SETTLE_MS` is a pause after each switch, so the clicks are
separate. If the bench is rewired, change the constants.

**What to do:** nothing.

In [ ]:
RELAY_BUS = 0  # STEMMA QT, scl=9 sda=8; the VL53L4CD is also here, at 0x29
RELAY_ADDR = 0x18
SETTLE_MS = 100

bus = bus_open("feathers3", RELAY_BUS, freq=100_000)
bus_lock = default_lock()


def relay_conn():
    """A connection to the relay on bus 0."""
    return I2CConnection(bus, RELAY_ADDR, None, bus_lock)


def rest():
    """Open the relay (command 0x00), whatever state it is in."""
    relay_conn().writeto(b"\x00")
    sleep_ms(SETTLE_MS)

## Find the part
Bus 0 should answer at `0x18` (the relay) and `0x29` (the VL53L4CD). If `0x18` is missing,
check the Qwiic cable, and whether the ADR jumper is closed (then the relay is at `0x19`).
This cell also opens the relay, in case an earlier run left it closed.

**What to do:** nothing.

In [ ]:
seen = bus.scan()
print("bus {} answers at:".format(RELAY_BUS), [hex(a) for a in seen])
check("relay ACKs at 0x18 on bus 0", RELAY_ADDR in seen)
others = [a for a in seen if a != RELAY_ADDR]
if others:
    info("also on bus 0: {}".format([hex(a) for a in others]))
if RELAY_ADDR in seen:
    rest()

## Matching tests: the fake's commands
The fake's unit tests that the real board can show too. `test_fake_refuses_a_read_without_a_reply`
has no match: the firmware's reply to an unknown command is not documented, and command `0x03`
changes the address, so the section does not send unknown commands.

**Matches** `test_fake_switches_on_commands`: raw command bytes, and the status after each.
`0x04` (a read command) leaves the relay as it is.

**What to do:** the relay clicks twice (on, off). Nothing must be wired to its contacts.

In [ ]:
conn = relay_conn()
try:
    states = []
    for cmd in (0x01, 0x01, 0x04, 0x00):
        conn.writeto(bytes([cmd]))
        sleep_ms(SETTLE_MS)
        states.append(conn.readfrom_mem(0x05, 1)[0] != 0)
    print("closed after 0x01, 0x01, 0x04, 0x00:", states)
    check("test_fake_switches_on_commands", states == [True, True, True, False])
finally:
    rest()

**Matches** `test_fake_reads_status_and_version_both_ways`: the "read byte" and a command
write followed by a separate read give the same status and version.

**What to do:** the relay clicks twice (on, off). Nothing must be wired to its contacts.

In [ ]:
conn = relay_conn()
try:
    conn.writeto(b"\x01")
    sleep_ms(SETTLE_MS)
    status_mem = conn.readfrom_mem(0x05, 1)
    conn.writeto(b"\x05")
    status_raw = conn.readfrom(1)
    version_mem = conn.readfrom_mem(0x04, 1)
    conn.writeto(b"\x04")
    version_raw = conn.readfrom(1)
    print("status: read byte {}, write-then-read {} | version: {}, {}".format(
        status_mem, status_raw, version_mem, version_raw))  # fmt: skip
    check("test_fake_reads_status_and_version_both_ways",
          status_mem == status_raw and status_mem[0] != 0 and version_mem == version_raw)  # fmt: skip
finally:
    rest()

## Matching tests: `QwiicRelay`

**Matches** `test_on_off_and_state`.

**What to do:** the relay clicks twice (on, off). Nothing must be wired to its contacts.

In [ ]:
relay = QwiicRelay(relay_conn())
try:
    before = relay.state
    relay.on()
    sleep_ms(SETTLE_MS)
    on = relay.state
    relay.off()
    sleep_ms(SETTLE_MS)
    off = relay.state
    print("state: before {}, after on() {}, after off() {}".format(before, on, off))
    check("test_on_off_and_state", (before, on, off) == (False, True, False))
finally:
    rest()

**Matches** `test_toggle_returns_the_new_state`.

**What to do:** the relay clicks three times (on, off, on), then once more to open. Nothing must
be wired to its contacts.

In [ ]:
relay = QwiicRelay(relay_conn())
try:
    relay.on()
    sleep_ms(SETTLE_MS)
    first = relay.toggle()
    sleep_ms(SETTLE_MS)
    second = relay.toggle()
    sleep_ms(SETTLE_MS)
    print("toggle() from on: {}, then {}".format(first, second))
    check("test_toggle_returns_the_new_state", (first, second) == (False, True))
finally:
    rest()

**Matches** `test_state_setter`.

**What to do:** the relay clicks twice (on, off). Nothing must be wired to its contacts.

In [ ]:
relay = QwiicRelay(relay_conn())
try:
    relay.state = True
    sleep_ms(SETTLE_MS)
    on = relay_conn().readfrom_mem(0x05, 1)[0] != 0
    relay.state = False
    sleep_ms(SETTLE_MS)
    off = relay_conn().readfrom_mem(0x05, 1)[0] != 0
    check("test_state_setter", on and not off)
finally:
    rest()

**Matches** `test_state_is_read_fresh`: the relay is closed with a raw command, behind the
driver's back, and `state` shows it at once.

**What to do:** the relay clicks twice (on, off). Nothing must be wired to its contacts.

In [ ]:
relay = QwiicRelay(relay_conn())
try:
    before = relay.state
    relay_conn().writeto(b"\x01")  # someone else switched it
    sleep_ms(SETTLE_MS)
    check("test_state_is_read_fresh", before is False and relay.state is True)
finally:
    rest()

**Matches** `test_firmware_version`. The real version is not known in advance: the check is
that it reads the same twice and is not `0xFF` (a floating bus).

**What to do:** nothing. The relay does not switch.

In [ ]:
relay = QwiicRelay(relay_conn())
v1 = relay.firmware_version()
v2 = relay.firmware_version()
print("firmware version: {:#04x}".format(v1))
check("test_firmware_version", v1 == v2 and v1 != 0xFF, "{:#04x}".format(v1))

**Matches** `test_address_and_clock_limits`.

**What to do:** nothing. No bus traffic.

In [ ]:
check("test_address_and_clock_limits",
      QwiicRelay.addr_ok(0x18) and QwiicRelay.addr_ok(0x19) and not QwiicRelay.addr_ok(0x20)
      and QwiicRelay.baud_ok(400_000) and not QwiicRelay.baud_ok(1_000_000))  # fmt: skip

`test_thermostat_holds_the_band_in_simulated_time` has no match here: there is no heater on the
bench.

## Hardware-only checks

Five toggles, `SETTLE_MS` apart: the state read back follows every one, and the relay ends
open.

**What to do:** the relay clicks five times, then once more to open. Nothing must be wired to
its contacts.

In [ ]:
relay = QwiicRelay(relay_conn())
try:
    got = []
    for _ in range(5):
        got.append(relay.toggle())
        sleep_ms(SETTLE_MS)
    print("states after each toggle:", got)
    check("five toggles followed", got == [True, False, True, False, True])
finally:
    rest()
check("relay left open", QwiicRelay(relay_conn()).state is False)

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail
the run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host, so it works
whether or not the board is still attached.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))